# 第 5 章习题与解答

> 本章习题聚焦 FFN、SwiGLU 和 MoE 的理解。建议先自己想,再展开答案。

## Exercise 5.1(易)

**题目**:把 SwiGLU 里的 `silu` 换成 `ReLU`,其他不变。用相同的随机输入跑一次 forward,观察输出有什么差异。为什么 SwiGLU 选择 SiLU 而不是 ReLU?

<details><summary><b>参考答案</b></summary>

把 `self.act_fn = F.silu` 改成 `self.act_fn = F.relu`,核心差异在于:

- **ReLU 版**:负区 gate 值恒为 0 → 门控完全关闭 → 信息被硬截断
- **SiLU 版**:负区 gate 值是小负值(如 x=-1 → -0.27)→ 门控部分保留 → 信息软性过滤

输出差异:ReLU 版的输出方差更小(因为更多维度被截断为 0),表达力下降。SiLU 处处可导、负区有梯度,在深层 Transformer 里训练更稳定。多篇论文(BERT→GELU, Llama→SiLU)验证了平滑激活优于 ReLU。

</details>

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math

d, d_ff = 768, math.ceil(768 * math.pi / 64) * 64

class SwiGLU(nn.Module):
    def __init__(self, act_fn):
        super().__init__()
        self.gate_proj = nn.Linear(d, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d, bias=False)
        self.up_proj   = nn.Linear(d, d_ff, bias=False)
        self.act_fn = act_fn
    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

torch.manual_seed(42)
x = torch.randn(4, 10, d)

# 两个模型用相同初始权重
silu_ffn = SwiGLU(F.silu)
relu_ffn = SwiGLU(F.relu)
relu_ffn.load_state_dict(silu_ffn.state_dict())  # 复制权重

y_silu = silu_ffn(x)
y_relu = relu_ffn(x)

print(f"SiLU 输出: mean={y_silu.mean():.4f}, std={y_silu.std():.4f}")
print(f"ReLU 输出: mean={y_relu.mean():.4f}, std={y_relu.std():.4f}")
print(f"差异 L2:   {(y_silu - y_relu).norm():.4f}")
print(f"→ ReLU 版输出更稀疏(std 更小),因为负区被硬截断")

## Exercise 5.2(中)

**题目**:手算 minimind 的 FFN 参数量。已知 `hidden_size=768`,`intermediate_size=2432`,SwiGLU 有 3 个矩阵(`gate_proj`, `up_proj`, `down_proj`),均无 bias。写出计算过程和最终结果。

<details><summary><b>参考答案</b></summary>

三个矩阵的尺寸:

- `gate_proj`: $768 \times 2432$(升维)
- `up_proj`: $768 \times 2432$(升维)
- `down_proj`: $2432 \times 768$(降维)

每个矩阵参数 = 行 × 列(无 bias):

$$P = 3 \times 768 \times 2432$$

逐项展开:

$$768 \times 2432 = 1{,}867{,}776$$

$$P = 3 \times 1{,}867{,}776 = 5{,}603{,}328$$

**每层 FFN 参数 = 5,603,328 ≈ 5.6M**。

对比:朴素 FFN(2 矩阵,4d 宽度)$= 2 \times 768 \times 3072 = 4{,}718{,}592$。SwiGLU 参数多了约 19%,但门控机制带来了更强的表达能力。

占每层总参数的比例:$5{,}603{,}328 / 7{,}374{,}528 \approx 76\%$。

</details>

In [ ]:
# 验算 Exercise 5.2
d, d_ff = 768, 2432

gate = d * d_ff      # 768 × 2432
up   = d * d_ff      # 768 × 2432
down = d_ff * d      # 2432 × 768
total = gate + up + down

print(f"gate_proj: {d} × {d_ff} = {gate:,}")
print(f"up_proj:   {d} × {d_ff} = {up:,}")
print(f"down_proj: {d_ff} × {d} = {down:,}")
print(f"总计:                   {total:,} ({total/1e6:.2f}M)")
print()

# 与标准 FFN 对比
std_ffn = 2 * d * (4 * d)
print(f"标准 FFN (2×768×3072): {std_ffn:,}")
print(f"SwiGLU/标准:           {total/std_ffn:.2f}× (参数多 19%)")

## Exercise 5.3(难)

**题目**:MoE 的 `aux_loss` 是如何实现负载均衡的?如果完全去掉 `aux_loss`(系数设为 0),训练会发生什么?

<details><summary><b>参考答案</b></summary>

**aux_loss 的机制**:

$$L_{\text{aux}} = \alpha \cdot N \cdot \sum_{i=1}^{N} \bar{f}_i \cdot \bar{P}_i$$

- $\bar{f}_i$:专家 $i$ 实际收到的 token **比例**(one-hot 统计)
- $\bar{P}_i$:router 给专家 $i$ 的平均**概率**(softmax 输出)

当负载完全均衡时,$\bar{f}_i = \bar{P}_i = 1/N$,$L_{\text{aux}} = N \cdot N \cdot (1/N)^2 = 1.0$(最小值)。

当负载不均衡时(比如所有 token 都给专家 0):$\bar{f}_0 = 1, \bar{f}_{others} = 0$,$L_{\text{aux}} = N \cdot \bar{P}_0$。由于 $\bar{P}_0$ 会因为赢者通吃而趋近 1,$L_{\text{aux}} \to N$(远大于 1)。

**梯度方向**:aux_loss 对 router 权重求导,会惩罚高 $\bar{f}_i \cdot \bar{P}_i$ 的专家,推动 router 把 token 分给负载低的专家。

**去掉 aux_loss 会发生什么**:

1. **赢者通吃**:训练初期某个专家碰巧效果好 → 梯度让它更强 → 更多 token 选它 → 其他专家收不到 token → 梯度为 0 → 永远不更新 → 「死专家」
2. **容量浪费**:4 个专家只有 1-2 个在工作,等于花钱买了 4 倍参数但只用 1/4
3. **模型退化**:MoE 退化成比 Dense 还差(因为 router 引入了噪声却没利用稀疏性)

这就是为什么 `router_aux_loss_coef = 5e-4` 虽然很小,但不可省略 —— 它是 MoE 训练的「保险丝」。

</details>

In [ ]:
# 演示 aux_loss 对负载均衡的影响
import torch, torch.nn.functional as F

num_experts = 4
num_tokens = 100

# 场景 1: 均衡路由(每个专家 25%)
balanced_routing = torch.tensor([25, 25, 25, 25], dtype=torch.float) / num_tokens
balanced_prob = torch.tensor([0.25, 0.25, 0.25, 0.25])

# 场景 2: 赢者通吃(专家 0 拿走所有)
collapsed_routing = torch.tensor([100, 0, 0, 0], dtype=torch.float) / num_tokens
collapsed_prob = torch.tensor([0.9, 0.04, 0.03, 0.03])  # router 也偏向专家 0

def aux_loss(frac, prob, N=4, alpha=5e-4):
    return alpha * N * (frac * prob).sum()

print("=== aux_loss 对比 ===")
print(f"均衡路由:    aux_loss = {aux_loss(balanced_routing, balanced_prob):.6f}  (理论最小 ≈ {5e-4 * 4 * 4 * 0.0625:.6f})")
print(f"赢者通吃:    aux_loss = {aux_loss(collapsed_routing, collapsed_prob):.6f}  (远大于均衡)")
print()

# 梯度方向演示
print("=== 不均衡时 aux_loss 的梯度方向 ===")
print("当专家 i 的 f_i · P_i 偏高时:")
print("  → aux_loss 增大")
print("  → ∂L_aux/∂router_weights > 0")
print("  → 梯度下降推动 router 减少给专家 i 的概率")
print("  → token 被重新分配给负载低的专家")
print()
print("结论: aux_loss 是 MoE 负载均衡的 '保险丝', 不可省略")